In [18]:
import geopandas as gpd
import pandas as pd

In [19]:
gdf = gpd.read_file("../data/chla_oan_gems_union.geojson")

In [20]:
gdf.columns

Index(['fecha', 'index_right', 'estacion', 'Decision', 'param', 'value',
       'unit', 'depth', 'granularidad', 'fuente', 'dist', 'nombre_programa',
       'id_estacion', 'nro_muestra', 'departamento', 'nombre_clave',
       'uni_nombre', 'valor_original', 'limite_deteccion',
       'limite_cuantificacion', 'valor_transformado', 'geometry'],
      dtype='str')

In [21]:
cols_2_remove = [
    "index_right",
    "nro_muestra",
    "uni_nombre",
    "granularidad",
    "depth",
    "dist",
    "param",
    "nombre_clave",
    "estacion",
    "id_estacion",
]

In [22]:
gdf = gdf.drop(columns=cols_2_remove)

In [23]:
gdf.head(5)

,fecha,Decision,value,unit,fuente,nombre_programa,departamento,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry
0,2017-01-02,NaN,NaN,NaN,OAN,Agua Montes del Plata Empresa,COLONIA,8.900,0.100,0.1,8.900,POINT (401164.986 6214585.025)
1,2017-01-02,si,0.0068,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,POINT (678380.002 6147956.963)
2,2017-01-02,si,0.0032,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,POINT (679045.962 6144090.994)
3,2017-01-02,NaN,NaN,NaN,OAN,Agua Montes del Plata Empresa,COLONIA,4.400,0.100,0.1,4.400,POINT (402233.023 6210401.038)
4,2017-01-02,NaN,NaN,NaN,OAN,Agua Montes del Plata Empresa,COLONIA,5.900,0.100,0.1,5.900,POINT (401555.004 6213022.975)


### Verificar que no hayan repeditos del output de : @union.py 

In [24]:
mediciones = gdf[["fecha", "valor_transformado", "value"]]
mediciones

,fecha,valor_transformado,value
0,2017-01-02 00:00:00,8.900,NaN
1,2017-01-02 00:00:00,NaN,0.0068
2,2017-01-02 00:00:00,NaN,0.0032
3,2017-01-02 00:00:00,4.400,NaN
4,2017-01-02 00:00:00,5.900,NaN
...,...,...,...
6518,2026-05-06 15:28:00,3.000000000,NaN
6519,2026-05-06 15:50:00,1.500000000,NaN
6520,2026-05-06 16:05:00,1.500000000,NaN
6521,2026-05-06 16:25:00,3.000000000,NaN


In [25]:
#for m in mediciones
gdf["fecha"].isna().sum()

#gdf["value"].isna().sum() == 

np.int64(0)

### Consolidando la columna de medicion
Este dataset 'chla_oan_gems_union.geojson' es la union entre las mediciones de gems y oan sin duplicaciones (originalmente muchos valoes de gems estaban duplicaods en OAN)

1. Las mediciones de GEMS se guardan en 'value'
2. Las medicones de OAN tienen la columna 'valor_original' y 'valor_transformado'
3. El objetivo es consolidar estas columnas bajo una sola columna 'chla' sin importar el origen de la medicon (OAN o GEMS)

In [26]:
num_faltante_oan = gdf[gdf["fuente"] == "OAN"]["valor_transformado"].isna().sum()
num_faltante_gems = gdf[gdf['fuente'] == 'GEMS']['value'].isna().sum()
print(f"Hay {num_faltante_gems} mediciones nulas de GEMS")
print(f"Hay {num_faltante_oan} mediciones nulas de OAN")

Hay 0 mediciones nulas de GEMS
Hay 9 mediciones nulas de OAN


In [27]:
gdf_cons = gdf.copy()
gdf_cons["chla"] = None # col vacia

In [28]:
# gems
gem_rows = gdf_cons["fuente"] == "GEMS"
gdf_cons.loc[gem_rows, "chla"] = gdf_cons.loc[gem_rows, "value"]*1000 # para que queden todos iguales 
#oan
oan_rows = gdf_cons["fuente"] == "OAN"
gdf_cons.loc[oan_rows, "chla"] = gdf_cons.loc[oan_rows, "valor_transformado"]

mg/m³ y µg/L son numéricamente idénticos. 1 mg/m³ = 1 µg/L, porque 1 m³ = 1000 L y 1 mg = 1000 µg

In [29]:
gdf_cons["chla"].isna().sum() # tiene sentido ya que solo 9 de OAN eran faltantes arriba

np.int64(9)

In [30]:
gdf_cons.head(10)

,fecha,Decision,value,unit,fuente,nombre_programa,departamento,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry,chla
0,2017-01-02,NaN,NaN,NaN,OAN,Agua Montes del Plata Empresa,COLONIA,8.900,0.100,0.1,8.900,POINT (401164.986 6214585.025),8.900
1,2017-01-02,si,0.0068,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,POINT (678380.002 6147956.963),6.8
2,2017-01-02,si,0.0032,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,POINT (679045.962 6144090.994),3.2
3,2017-01-02,NaN,NaN,NaN,OAN,Agua Montes del Plata Empresa,COLONIA,4.400,0.100,0.1,4.400,POINT (402233.023 6210401.038),4.400
4,2017-01-02,NaN,NaN,NaN,OAN,Agua Montes del Plata Empresa,COLONIA,5.900,0.100,0.1,5.900,POINT (401555.004 6213022.975),5.900
5,2017-01-02,si,0.0059,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,POINT (676618.974 6142887.026),5.9
6,2017-01-02,si,0.0114,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,POINT (669756.975 6142832.05),11.4
7,2017-01-02,NaN,NaN,NaN,OAN,Agua Montes del Plata Empresa,COLONIA,4.400,0.100,0.1,4.400,POINT (402625.965 6210316.961),4.400
8,2017-01-02,dudoso,0.0059,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,POINT (675890.225 6151378.242),5.9
9,2017-01-02,si,0.0104,mg/l,GEMS,NaN,NaN,NaN,NaN,NaN,NaN,POINT (410964.981 6209417.971),10.4


#### Hagamos una verificacion mas de que nuestra deduplicacion funciono correctamente.

Si nuestra dedup funcion bien entonces:
1. cada 3-tuple (location, fecha, value) tiene que ser unica

In [31]:
uniques = set()
for _, r in gdf_cons.iterrows():
    sample = (r["geometry"], r["fecha"], r["chla"])

    if pd.isna(sample[2]):
        # tenemos 9 nulos de OAN, ignoramos esto por ahora proabalemente los vamos a quitar
        continue

    if sample in uniques:
        print("Sample repetida :(")
        break
    uniques.add(sample)

assert len(uniques) == len(gdf) - num_faltante_oan
print(f"No hay samples repetidas :)")

No hay samples repetidas :)


In [32]:
#Borrar valores nuelos:
gdf_cons = gdf_cons.dropna(subset=["chla"])

In [33]:
gdf_cons["chla"].isna().sum()

np.int64(0)

In [34]:
gdf_cons.to_file("../data/oan_gems_clean.geojson", driver="GeoJSON")